# Roller Compaction DOE/RSM — Sample Analysis

**Dataset:** Roller Compaction: Ribbon Density vs. Process Parameters (Synthetic) v1.0
**Publisher:** [Innovative Process Applications (IPA)](https://www.innovativeprocess.com)
**License:** CC BY 4.0

> This is **synthetic educational data**. See `README.md` for the physical model (Johanson + Heckel) used to generate it.

This notebook walks through a typical Quality-by-Design workflow:
1. Load and explore the data
2. Fit a response surface (quadratic) model for ribbon density
3. Identify the operating window that maximizes density while keeping uniformity (CV%) under 3%

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.metrics import r2_score

df = pd.read_csv('ribbon_density_v1.0.csv')
print(df.shape)
df.head()

## 1. Explore the process parameters

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
axes[0].scatter(df['roll_force_kN_per_cm'], df['ribbon_rel_density'], alpha=0.4)
axes[0].set_xlabel('Roll force (kN/cm)'); axes[0].set_ylabel('Relative density')
axes[0].set_title('Force → density (Heckel)')

axes[1].scatter(df['feed_screw_rpm']/df['roll_speed_rpm'], df['density_CV_percent'], alpha=0.4, color='C1')
axes[1].set_xlabel('Feed/roll speed ratio'); axes[1].set_ylabel('Density CV %')
axes[1].set_title('Feed ratio → uniformity')

axes[2].scatter(df['peak_pressure_MPa'], df['ribbon_rel_density'], alpha=0.4, color='C2')
axes[2].set_xlabel('Peak nip pressure (MPa)'); axes[2].set_ylabel('Relative density')
axes[2].set_title('Pressure → density')
plt.tight_layout(); plt.show()

## 2. Fit a quadratic response surface

Classic RSM model: ribbon density as a quadratic function of the three main process parameters.

In [ ]:
X = df[['roll_force_kN_per_cm', 'roll_speed_rpm', 'feed_screw_rpm']].values
y = df['ribbon_rel_density'].values

poly = PolynomialFeatures(degree=2, include_bias=False)
Xp = poly.fit_transform(X)
model = LinearRegression().fit(Xp, y)
y_pred = model.predict(Xp)
print(f'R² = {r2_score(y, y_pred):.3f}')

plt.figure(figsize=(5,5))
plt.scatter(y, y_pred, alpha=0.4)
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'k--')
plt.xlabel('Observed rel. density'); plt.ylabel('Predicted')
plt.title('RSM fit'); plt.show()

## 3. Find the design space

QbD asks: which combinations of process parameters give us a target density **AND** acceptable uniformity?

Let's define an acceptable region as: relative density ≥ 0.70 AND density CV% ≤ 3.0%.

In [ ]:
in_spec = (df['ribbon_rel_density'] >= 0.70) & (df['density_CV_percent'] <= 3.0)
print(f'{in_spec.sum()} of {len(df)} runs meet spec ({100*in_spec.mean():.1f}%)')

fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(df.loc[~in_spec, 'roll_force_kN_per_cm'], df.loc[~in_spec, 'feed_screw_rpm']/df.loc[~in_spec, 'roll_speed_rpm'], alpha=0.3, label='Out of spec', color='lightgray')
ax.scatter(df.loc[in_spec, 'roll_force_kN_per_cm'], df.loc[in_spec, 'feed_screw_rpm']/df.loc[in_spec, 'roll_speed_rpm'], alpha=0.6, label='In spec', color='teal')
ax.set_xlabel('Roll force (kN/cm)'); ax.set_ylabel('Feed/roll ratio')
ax.set_title('Design space: density ≥ 0.70 AND CV ≤ 3%')
ax.legend(); plt.show()

## Takeaway

The design space clusters around **roll force ≥ 6 kN/cm** and **feed/roll ratio ≈ 8–15** — consistent with the physics of a well-fed nip on a twin-feed-screw roller compactor.

For production-scale equipment that achieves this operating window consistently, see IPA's CL-series twin-feed-screw compactors: https://www.innovativeprocess.com

---
*Dataset © 2026 Innovative Process Applications, CC BY 4.0. Synthetic educational data — not real measurements.*